# Tarefa 03

- Leia os enunciados com atenção
- Saiba que pode haver mais de uma resposta correta
- Insira novas células de código sempre que achar necessário
- Em caso de dúvidas, procure os Tutores
- Divirta-se :)

In [1]:
import pandas as pd
import requests

####  1) Lendo de APIs
Vimos em aula como carregar dados públicos do governo através de um API (*Application Programming Interface*). No exemplo de aula, baixamos os dados de pedidos de verificação de limites (PVL) realizados por estados, e selecionamos apenas aqueles referentes ao estado de São Paulo.

1. Repita os mesmos passos feitos em aula, mas selecione os PVLs realizados por municípios no estado do Rio de Janeiro.
2. Quais são os três *status* das solicitações mais frequentes na base? Quais são suas frequências?
3. Construa uma nova variável que contenha o ano do **status**. Observe que ```data_status``` vem como tipo *object* no **DataFrame**. Dica: você pode usar o método ```.str``` para transformar o tipo da variável em string, em seguida um método como [**slice()**](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.slice.html) ou [**split()**](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.str.split.html).
4. Indique a frequência de cada ano do campo construído no item (3).

In [15]:
#1) Seu código aqui

api = 'https://apidatalake.tesouro.gov.br/ords/sadipem/tt/pvl'
verificar = requests.get(api)
verificar.status_code
dados = verificar.json()
pd.DataFrame(dados['items'])
#tudo ok até senhores

df = pd.DataFrame(dados['items']) # Cria um DataFrame com base nos itens retornados pela API
pvl_rj = df[df['uf'] == 'RJ'] # Filtra os PVLs realizados por municípios no estado do Rio de Janeiro
print(pvl_rj) #Exibe o DataFrame resultante  
    




      id_pleito tipo_interessado            interessado  cod_ibge  uf  \
81         5290        Município      Casimiro de Abreu   3301306  RJ   
277       61009        Município         Rio das Flores   3304508  RJ   
353       66228        Município    São Pedro da Aldeia   3305208  RJ   
846        6363        Município              Rio Claro   3304409  RJ   
892        7907        Município               Mesquita   3302858  RJ   
...         ...              ...                    ...       ...  ..   
4650       9898           Estado         Rio de Janeiro        33  RJ   
4707      13229        Município          Nova Friburgo   3303401  RJ   
4917      38893        Município        Paty do Alferes   3303856  RJ   
4945      11338        Município        Duque de Caxias   3301702  RJ   
4957       9205        Município  Campos dos Goytacazes   3301009  RJ   

                   num_pvl                          status  \
81                    None                        Deferido   

In [6]:
# 2) Seu código aqui
contagem_status = pvl_rj['status'].value_counts()

# Exibe os três status mais frequentes e suas frequências
top_3_status = contagem_status.head(3)
print("Os três status mais frequentes e suas frequências são:")
print(top_3_status)



Os três status mais frequentes e suas frequências são:
status
Deferido        24
Arquivado       11
Regularizado    10
Name: count, dtype: int64


In [18]:
# 3) Seu código aqui
# Filtra os itens com "tipo_interessado" igual a "Município" e "uf" igual a "BA"
filtro_municipio_ba = (df['tipo_interessado'] == 'Município') & (df['uf'] == 'BA')
itens_municipio_ba = df[filtro_municipio_ba]

# Identifica o item com mais status "Deferido" entre os itens restantes
item_mais_deferido = itens_municipio_ba.loc[itens_municipio_ba['status'] == 'Deferido'].max()

# Imprime o resultado
print("Itens com tipo_interessado 'Município' e uf 'BA':")
print(itens_municipio_ba)

print("\nItem com mais status 'Deferido':")
print(item_mais_deferido)

TypeError: '>=' not supported between instances of 'str' and 'float'

####  2) Melhorando a interação com o API
Observe dois URLs de consultas diferentes, por exemplo o URL utilizado em aula, e o URL feito no exercício anterior. Compare-os e observe as diferenças.

1. Faça uma função em Python que recebe como argumento o UF da consulta e o tipo de interessado (```'Estado'```ou ```Município```), e que devolve os dados da consulta no formato *DataFrame*.
2. Quantas solicitações para o Estado podem ser consultadas para Minas Gerais com *status* em 'Arquivado por decurso de prazo' estão registradas?
3. Qual é o município da Bahia com mais solicitações deferidas?
4. Salve um arquivo .csv com os dados de solicitações da Bahia, com interessado = 'Estado'

In [10]:
#1) Seu código aqui

def consultar_dados_pvl(uf, tipo_interessado):
    if verificar.status_code == 200:
        dados = verificar.json()
        df = pd.DataFrame(dados['items'])# Cria um DataFrame com base nos itens retornados pela API
        filtro = (df['uf'] == uf) & (df['tipo_interessado'] == tipo_interessado)# Filtra os PVLs com base no UF e tipo de interessado
        resultado = df[filtro]
        return resultado
    else:
        print(f"A requisição falhou com o status code {verificar.status_code}")
        return None
    
#Testando a função    
uf_consulta = 'RJ'
tipo_interessado_consulta = 'Município'
resultado_consulta = consultar_dados_pvl(uf_consulta, tipo_interessado_consulta)
print(resultado_consulta)
#Funcionou doidão

      id_pleito tipo_interessado            interessado  cod_ibge  uf  \
81         5290        Município      Casimiro de Abreu   3301306  RJ   
277       61009        Município         Rio das Flores   3304508  RJ   
353       66228        Município    São Pedro da Aldeia   3305208  RJ   
846        6363        Município              Rio Claro   3304409  RJ   
892        7907        Município               Mesquita   3302858  RJ   
936        9109        Município            Barra Mansa   3300407  RJ   
1024       7216        Município     São João de Meriti   3305109  RJ   
1030       7596        Município            Barra Mansa   3300407  RJ   
1081       9317        Município            Nova Iguaçu   3303500  RJ   
1093      10174        Município               Miracema   3303005  RJ   
1157       9970        Município              Queimados   3304144  RJ   
1367       8643        Município         Rio de Janeiro   3304557  RJ   
1396       8050        Município         Rio de Jan

In [11]:
# 2) Seu código aqui
import requests
import pandas as pd

def contar_solicitacoes_arquivadas(uf, status):
    if verificar.status_code == 200:
        dados = verificar.json()# Converte a resposta para formato JSON        
        df = pd.DataFrame(dados['items'])# Cria um DataFrame com base nos itens retornados pela API        
        filtro = (df['uf'] == uf) & (df['status'] == status)# Filtra os PVLs com base no UF e status
        resultado = df[filtro]        
        return len(resultado)# Retorna o número de solicitações encontradas

    else:
        print(f"A requisição falhou com o status code {verificar.status_code}")
        return None

# Testando
uf_consulta = 'MG'
status_consulta = 'Arquivado por decurso de prazo'
quantidade_solicitacoes = contar_solicitacoes_arquivadas(uf_consulta, status_consulta)

# Deu
print(f"Quantidade de solicitações para {uf_consulta} com status '{status_consulta}': {quantidade_solicitacoes}")

Quantidade de solicitações para MG com status 'Arquivado por decurso de prazo': 73


In [20]:
# 3) Seu código aqui
# Filtra os itens com "tipo_interessado" igual a "Município" e "uf" igual a "BA" e "status" igual a "Deferido"
filtro_municipio_ba_deferido = (df['tipo_interessado'] == 'Município') & (df['uf'] == 'BA') & (df['status'] == 'Deferido')
itens_municipio_ba_deferido = df[filtro_municipio_ba_deferido]

# Identifica o município com mais solicitações deferidas
municipio_mais_deferido = itens_municipio_ba_deferido['interessado'].value_counts().idxmax()

# Conta o número de solicitações deferidas para o município mais frequente
quantidade_solicitacoes_deferidas_municipio_mais_frequente = itens_municipio_ba_deferido[itens_municipio_ba_deferido['interessado'] == municipio_mais_deferido].shape[0]

# Imprime o resultado
print(f'O município da Bahia com mais solicitações deferidas é {municipio_mais_deferido}, com um total de {quantidade_solicitacoes_deferidas_municipio_mais_frequente} solicitações deferidas.')

O município da Bahia com mais solicitações deferidas é Lauro de Freitas, com um total de 3 solicitações deferidas.


In [21]:
# 4) Seu código aqui
filtro_estado_ba = (df['uf'] == 'BA') & (df['interessado'] == 'Estado')
solicitacoes_estado_ba = df[filtro_estado_ba]

# Salva o DataFrame filtrado em um arquivo CSV
solicitacoes_estado_ba.to_csv('solicitacoes_estado_ba.csv', index=False)

print("Arquivo CSV salvo com sucesso.")

Arquivo CSV salvo com sucesso.
